# In Class Activity April 14th, 2026

### Importing libraries, preparing data, initial EDA

In [ ]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, RepeatedStratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [2]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [5]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





From my EDA, it was pretty obvious some features mattered way more than others, so I didn’t just use everything. I also noticed the classes weren’t evenly distributed, which is something to keep in mind when looking at results.
A lot of the variables were kind of skewed and there were a few outliers, so using tree-based models like Random Forest and XGBoost just made more sense. Overall, EDA mostly just helped me understand what was going on in the data and make better calls when building the models.

### Data Preprocessing (minimal) and Baseline Model

In [3]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [4]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [5]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline XGBoost model with default settings gave a mean cross-validated F1 score of 0.712. Since the dataset is imbalanced (more negative than positive income classes), this score could be better. To improve it, I'll experiment with XGBoost's hyperparameters like scale_pos_weight to handle imbalance, max_depth for tree complexity, and learning_rate for convergence. I'll also try different tuning approaches to find the best combination.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [11]:
from sklearn.model_selection import RepeatedStratifiedKFold

# explore several XGBoost hyperparameters with repeated stratified cross validation
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

def evaluate_xgb(name, **params):
    """Train an XGBoost model with the given parameters and return repeated CV F1 scores."""
    model = XGBClassifier(enable_categorical=True,
                          random_state=42,
                          eval_metric='logloss',
                          **params)
    scores = cross_val_score(model, X_train, y_train, cv=rskf, scoring='f1', n_jobs=-1)
    print(f'{name}: mean F1 = {scores.mean():.4f}, std = {scores.std():.4f}')
    return scores.mean(), scores.std()

experiments = [
    ('Baseline', {}),
    ('Higher max_depth', {'max_depth': 8}),
    ('Lower learning_rate with more estimators', {'learning_rate': 0.05, 'n_estimators': 300}),
    ('Scale_pos_weight adjusted for imbalance', {'scale_pos_weight': y_train.value_counts()[0] / y_train.value_counts()[1]}),
    ('Subsample + colsample_bytree regularization', {'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 1, 'reg_lambda': 2})
]

results = []
for name, params in experiments:
    mean_f1, std_f1 = evaluate_xgb(name, **params)
    results.append((name, params, mean_f1, std_f1))

best_name, best_params, best_mean, best_std = max(results, key=lambda x: x[2])
print('\nBest model based on repeated CV F1:')
print(f'  {best_name} with mean F1 = {best_mean:.4f} and std = {best_std:.4f}')

best_model = XGBClassifier(enable_categorical=True,
                           random_state=42,
                           use_label_encoder=False,
                           eval_metric='logloss',
                           **best_params)
best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)
print('\nClassification report for the best model on the test set:')
print(classification_report(y_test, y_pred_best))

Baseline: mean F1 = 0.7070, std = 0.0092
Higher max_depth: mean F1 = 0.7020, std = 0.0103
Lower learning_rate with more estimators: mean F1 = 0.7109, std = 0.0098
Scale_pos_weight adjusted for imbalance: mean F1 = 0.7127, std = 0.0082
Subsample + colsample_bytree regularization: mean F1 = 0.7038, std = 0.0103

Best model based on repeated CV F1:
  Scale_pos_weight adjusted for imbalance with mean F1 = 0.7127 and std = 0.0082


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/xgboost/training.py:200: UserWarning: [14:52:33] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Classification report for the best model on the test set:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7431
           1       0.62      0.84      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.78      0.84      0.80      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [12]:
# set up GridSearchCV for the preferred model from above
param_grid = {
    'scale_pos_weight': [1.0, y_train.value_counts()[0] / y_train.value_counts()[1], 5.0, 10.0],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

base_model = XGBClassifier(random_state=42, enable_categorical=True, eval_metric='logloss')

grid_search = GridSearchCV(base_model,
                           param_grid=param_grid,
                           scoring='f1',
                           cv=skf,
                           n_jobs=-1,
                           verbose=1)

grid_search.fit(X_train, y_train)

print('Best parameters from GridSearchCV:', grid_search.best_params_)
print('Best CV F1 score from GridSearchCV:', grid_search.best_score_)

best_grid_model = XGBClassifier(random_state=42,
                                enable_categorical=True,
                                eval_metric='logloss',
                                **grid_search.best_params_)
best_grid_model.fit(X_train, y_train)

y_pred_grid = best_grid_model.predict(X_test)
print('\nGridSearchCV-tuned model performance on test set:')
print(classification_report(y_test, y_pred_grid))

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best parameters from GridSearchCV: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 8, 'scale_pos_weight': 3.1793774735265803, 'subsample': 0.8}
Best CV F1 score from GridSearchCV: 0.716845025114193

GridSearchCV-tuned model performance on test set:
              precision    recall  f1-score   support

           0       0.95      0.84      0.89      7431
           1       0.62      0.86      0.72      2338

    accuracy                           0.84      9769
   macro avg       0.78      0.85      0.80      9769
weighted avg       0.87      0.84      0.85      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [13]:
# tuning xgboost classifier with RandomizedSearchCV using the preferred base model from above
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(4, 11),
    'learning_rate': np.linspace(0.01, 0.2, num=10),
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

base_model = XGBClassifier(random_state=42, enable_categorical=True, eval_metric='logloss')

xgb_random = RandomizedSearchCV(base_model,
                                param_distributions=param_dist, n_iter=30, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
xgb_random_best = XGBClassifier(random_state=42,
                                scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'], 
                                subsample=xgb_random.best_params_['subsample'],
                                colsample_bytree=xgb_random.best_params_['colsample_bytree'],
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)

y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')

Best parameters from RandomizedSearchCV: {'subsample': 1.0, 'scale_pos_weight': 2.0, 'max_depth': 6, 'learning_rate': 0.1577777777777778, 'colsample_bytree': 0.7}
Best F1 score from RandomizedSearchCV: 0.7280264139770931
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.68      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [16]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 4, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2)
    subsample = trial.suggest_float('subsample', 0.7, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 1.0)
    
    # use the preferred base model from above and tune the same hyperparameters
    xgb_optuna = XGBClassifier(random_state=42,
                               enable_categorical=True,
                               eval_metric='logloss',
                               scale_pos_weight=scale_pos_weight,
                               max_depth=max_depth,
                               learning_rate=learning_rate,
                               subsample=subsample,
                               colsample_bytree=colsample_bytree)
    
    cv_scores = cross_val_score(xgb_optuna, X_train, y_train, cv=rskf, scoring='f1', n_jobs=-1)
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42,
                                enable_categorical=True,
                                eval_metric='logloss',
                                scale_pos_weight=study.best_params['scale_pos_weight'],
                                max_depth=study.best_params['max_depth'],
                                learning_rate=study.best_params['learning_rate'],
                                subsample=study.best_params['subsample'],
                                colsample_bytree=study.best_params['colsample_bytree'])

xgb_optuna_best.fit(X_train, y_train)

y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')

[I 2026-04-15 15:12:26,495] A new study created in memory with name: no-name-012f3dcb-8781-40b4-b7b0-0f80a9bad2c8
[I 2026-04-15 15:12:43,669] Trial 0 finished with value: 0.7007250030976174 and parameters: {'scale_pos_weight': 3.321180786375118, 'max_depth': 10, 'learning_rate': 0.01605723971782286, 'subsample': 0.9754988629014568, 'colsample_bytree': 0.9850379621772987}. Best is trial 0 with value: 0.7007250030976174.
[I 2026-04-15 15:12:45,882] Trial 1 finished with value: 0.6643119890203358 and parameters: {'scale_pos_weight': 8.129206294315178, 'max_depth': 5, 'learning_rate': 0.12395600568284885, 'subsample': 0.8305529393815481, 'colsample_bytree': 0.9465327667929258}. Best is trial 0 with value: 0.7007250030976174.
[I 2026-04-15 15:12:50,256] Trial 2 finished with value: 0.6906161471196494 and parameters: {'scale_pos_weight': 8.450091965238821, 'max_depth': 9, 'learning_rate': 0.1390763148145725, 'subsample': 0.800380505020031, 'colsample_bytree': 0.9481475911950193}. Best is tri

Best parameters from Optuna: {'scale_pos_weight': 1.902308087989784, 'max_depth': 8, 'learning_rate': 0.10307080784988032, 'subsample': 0.7913242115506882, 'colsample_bytree': 0.7932143829683129}
Best F1 score from Optuna: 0.7266768131530156
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.89      0.91      7431
           1       0.69      0.78      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.81      0.83      0.82      9769
weighted avg       0.87      0.86      0.87      9769



### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


Tuning XGBoost hyperparameters was actually pretty interesting. I tried out GridSearchCV, RandomizedSearchCV, and Optuna, and RandomizedSearchCV ended up working the best for me. It gave me a CV F1 score of 0.7280 and a test F1 of 0.73 for the minority class.
It was cool to see how the different methods compare. Grid search felt a bit slow and rigid, while randomized search was way more efficient and still got strong results. I didn’t go super deep with Optuna this time, but I can see why people like it since it seems a lot more flexible. I’d probably use RandomizedSearchCV again for something like this, but I want to try Optuna more next time.